# Download the full Kerr run from Google Drive to your PC

Run this **in Colab**. It grabs two things:
1. **The run output** `fno_hybrid_kerr` (model.pt + per_sample.json + report.json + figs) — small (~tens of MB).
2. **The test dataset fields** `dataset_test.npz` — large (~1-3 GB), only needed to regenerate the ringdown / pointwise-error plots.

Each file is saved to your Downloads folder via the browser.

In [ ]:
# 1. Mount Drive and show what's in the run folder
import os
from google.colab import drive
drive.mount('/content/drive')

RUN_DIR = '/content/drive/MyDrive/kerr_l4/fno_hybrid_kerr'
print('run dir exists:', os.path.isdir(RUN_DIR))
for root, _, files in os.walk(RUN_DIR):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f'{os.path.getsize(p)/1e6:9.2f} MB  {os.path.relpath(p, RUN_DIR)}')

In [ ]:
# 2. Zip the run output (model + json + figs) and download it
import shutil
from google.colab import files

zip_base = '/content/fno_hybrid_kerr'
shutil.make_archive(zip_base, 'zip', RUN_DIR)
print('zip size:', round(os.path.getsize(zip_base + '.zip') / 1e6, 1), 'MB')
files.download(zip_base + '.zip')

In [ ]:
# 3. Find the test-split raw fields (needed to re-plot ringdown / pointwise-error).
#    Searches the current Colab session AND Drive.
import glob

cands = (glob.glob('/content/**/phase_c_l4_decoupled/dataset_test.npz', recursive=True)
         + glob.glob('/content/drive/MyDrive/**/dataset_test.npz', recursive=True))
cands = sorted(set(cands))
for c in cands:
    print(f'{os.path.getsize(c)/1e9:6.2f} GB  {c}')
if not cands:
    print('NOT FOUND in this session. The dataset lives in the working dir that is')
    print('wiped when the runtime resets. Re-run the training/eval session, or copy')
    print('phase_c_l4_decoupled/ to Drive, then re-run this cell.')

In [ ]:
# 4. Download the test dataset (LARGE). Direct browser download can be flaky for
#    multi-GB files, so also copy it to Drive as a reliable fallback.
if cands:
    src = cands[0]
    dst = '/content/drive/MyDrive/kerr_l4/dataset_test.npz'
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        print('copied to Drive:', dst)
    print('attempting browser download (may be slow for >1 GB) ...')
    from google.colab import files
    files.download(src)
else:
    print('No dataset_test.npz found — see cell 3.')